In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import Adam, SGD
import keras_tuner as kt
import datetime

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [ ]:
X_train = X_train.reshape(-1, 28*28)
X_val = X_val.reshape(-1, 28*28)
X_test = X_test.reshape(-1, 28*28)

In [ ]:
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
baseline_model = Sequential([
    Dense(128, activation='relu', input_shape=(28*28,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('baseline_model.h5', save_best_only=True)
log_dir = "logs/baseline_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_cb = TensorBoard(log_dir=log_dir)

# Train
history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, checkpoint, tensorboard_cb]
)

# Evaluate
test_loss, test_acc = baseline_model.evaluate(X_test, y_test)
print("Baseline Test Accuracy:", test_acc)

In [ ]:
plt.plot(history_baseline.history['accuracy'], label='train')
plt.plot(history_baseline.history['val_accuracy'], label='val')
plt.title('Baseline Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(
        units=hp.Int('units1', min_value=32, max_value=256, step=32),
        activation='relu',
        input_shape=(28*28,)
    ))

    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(Dense(
            units=hp.Int(f'units_{i+2}', min_value=32, max_value=256, step=32),
            activation='relu'
        ))
        if hp.Boolean('dropout'):
            model.add(Dropout(rate=hp.Float('dropout_rate', 0.1, 0.5, step=0.1)))

    model.add(Dense(10, activation='softmax'))

    optimizer = hp.Choice('optimizer', ['adam', 'sgd'])
    if optimizer == 'adam':
        opt = Adam(learning_rate=hp.Float('lr', 1e-4, 1e-2, sampling='log'))
    else:
        opt = SGD(learning_rate=hp.Float('lr', 1e-4, 1e-2, sampling='log'))

    model.compile(optimizer=opt,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='tuner_logs',
    project_name='fashion_mnist'
)

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)]
)

best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(1)[0]

In [ ]:
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print("Tuned Model Test Accuracy:", test_acc)

The number of hidden layers and neurons per layer had the largest impact on model performance. Using dropout slightly improved generalization. Automated tuning achieved a higher test accuracy than manual tuning, and it saved time exploring the hyperparameter space systematically. I learned that deep MLPs can quickly overfit without proper regularization and early stopping.